In [1]:
import pandas as pd
import numpy as np

In [13]:
final_df = pd.read_csv('Final_Dataset.csv')

# Create a copy for model training
df = final_df.copy()

# Inspect the dataset
# print("Dataset shape:", df.shape)
# print("\nMissing values:")
# print(df.isnull().sum()[df.isnull().sum() > 0])

# print("\nTarget summary:")
# print(df['Values'].describe())

df.columns

Index(['Unnamed: 0', 'Player', 'Position', 'Team', 'Comp', 'Age', 'MP', 'Min',
       'Goals', 'Ast', 'G+A', 'xG', 'xA', 'SoT%', 'npxG/Sh', 'G-xG',
       'Pass Cmp%', 'Through Balls', 'Switch Passes', 'Crosses',
       'Shot Creating Actions', 'Def Action to Shot', 'Tackles',
       'Shots Blocked', 'Interceptions', 'Clearances', 'Errors',
       'Dribbles Successful', 'Dribble Distance', 'Mistakes',
       'Ball Recoveries', 'Air Duels Won%', 'Values'],
      dtype='str')

In [14]:
# Separating input features and target
X = final_df.drop(columns=['Values', 'Player', 'Team', 'Comp'], errors='ignore')
y = final_df['Values'].copy()

print("Feature shape:", X.shape)
print("Target shape:", y.shape)
print("Features:", X.columns.tolist())

Feature shape: (1989, 29)
Target shape: (1989,)
Features: ['Unnamed: 0', 'Position', 'Age', 'MP', 'Min', 'Goals', 'Ast', 'G+A', 'xG', 'xA', 'SoT%', 'npxG/Sh', 'G-xG', 'Pass Cmp%', 'Through Balls', 'Switch Passes', 'Crosses', 'Shot Creating Actions', 'Def Action to Shot', 'Tackles', 'Shots Blocked', 'Interceptions', 'Clearances', 'Errors', 'Dribbles Successful', 'Dribble Distance', 'Mistakes', 'Ball Recoveries', 'Air Duels Won%']


In [ ]:
# Add Value Team
final_df['Value Team'] = (
    final_df.groupby('Team')['Values'].transform('sum')
    - final_df['Values']
)

# Recreate features and target
X = final_df.drop(
    columns=['Values', 'Player', 'Team', 'Comp', 'Unnamed: 0'],
    errors='ignore'
)
y = final_df['Values'].copy()

print("Value Team added:", 'Value Team' in X.columns)
print("Feature shape:", X.shape)

Value Team added: True
Feature shape: (1989, 29)


,Position,Age,MP,Min,Goals,Ast,G+A,xG,xA,SoT%,...,Shots Blocked,Interceptions,Clearances,Errors,Dribbles Successful,Dribble Distance,Mistakes,Ball Recoveries,Air Duels Won%,Value Team
0,DF,27-175,23,2070,0,9,9,0.6,3.0,15.4,...,13,44,75,0,25,7034,27,96,61.4,287000000.0
1,DF,23-275,23,2070,2,1,3,2.2,1.1,41.7,...,12,20,130,2,5,7142,10,105,65.2,310900000.0
2,DF,22-151,23,2070,0,0,0,0.8,0.4,0.0,...,7,23,117,2,4,7181,10,109,67.9,322000000.0
3,FW,25-176,23,2064,14,3,17,7.7,5.1,53.7,...,41,9,34,0,29,2570,65,95,29.7,288900000.0
4,MF,23-089,23,2035,1,2,3,0.6,1.7,16.7,...,12,33,36,0,14,5574,23,137,70.6,796000000.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1984,DF,25-325,2,90,0,0,0,0.0,0.0,0.0,...,0,1,1,0,1,201,1,5,100.0,167500000.0
1985,DF,25-329,1,90,0,0,0,0.0,0.1,0.0,...,0,2,2,0,0,677,0,1,61.5,923500000.0
1986,DF,20-035,1,90,0,0,0,0.0,0.0,0.0,...,0,1,3,0,0,111,2,6,0.0,312500000.0
1987,DF,21-227,1,90,0,0,0,0.0,0.0,0.0,...,0,2,0,0,0,568,1,4,33.3,386650000.0


In [5]:
from sklearn.model_selection import train_test_split

# 60% training, 20% validation, 20% testing
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

print("Training set:", X_train.shape)
print("Validation set:", X_val.shape)
print("Test set:", X_test.shape)

Training set: (1193, 29)
Validation set: (398, 29)
Test set: (398, 29)


In [6]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# Convert Age to numeric if stored in formats like "27-175"
X_train = X_train.copy()
X_val = X_val.copy()
X_test = X_test.copy()

for dataset in [X_train, X_val, X_test]:
    if 'Age' in dataset.columns:
        dataset['Age'] = pd.to_numeric(
            dataset['Age'].astype(str).str.split('-').str[0],
            errors='coerce'
        )

# Identify categorical and numerical columns
categorical_features = X_train.select_dtypes(
    include=['object', 'str', 'category', 'string']
).columns.tolist()

numerical_features = X_train.select_dtypes(
    include=np.number
).columns.tolist()

# Preprocessing pipelines
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numerical_features),
    ('cat', categorical_transformer, categorical_features)
])

print("Numerical features:", len(numerical_features))
print("Categorical features:", categorical_features)

Numerical features: 28
Categorical features: ['Position']


In [7]:
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    ExtraTreesRegressor
)
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

models = {
    "Random Forest": RandomForestRegressor(
        n_estimators=200, random_state=42, n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
    "Extra Trees": ExtraTreesRegressor(
        n_estimators=200, random_state=42, n_jobs=-1
    )
}

results = []

for name, model in models.items():
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_val)

    results.append({
        "Model": name,
        "R²": r2_score(y_val, predictions),
        "MAE": mean_absolute_error(y_val, predictions),
        "RMSE": np.sqrt(mean_squared_error(y_val, predictions))
    })

results_df = pd.DataFrame(results).sort_values("R²", ascending=False)
print(results_df.to_string(index=False))

            Model       R²          MAE         RMSE
    Random Forest 0.701658 5.986553e+06 9.348495e+06
Gradient Boosting 0.695766 5.925018e+06 9.440353e+06
      Extra Trees 0.679510 6.268310e+06 9.689275e+06


In [8]:
# Train the same models using log-transformed market values
log_results = []

for name, model in models.items():
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipeline.fit(X_train, np.log1p(y_train))

    log_predictions = np.expm1(pipeline.predict(X_val))
    log_predictions = np.maximum(log_predictions, 0)

    log_results.append({
        "Model": name,
        "R²": r2_score(y_val, log_predictions),
        "MAE": mean_absolute_error(y_val, log_predictions),
        "RMSE": np.sqrt(mean_squared_error(y_val, log_predictions))
    })

log_results_df = pd.DataFrame(log_results).sort_values(
    "R²", ascending=False
)

print(log_results_df.to_string(index=False))

            Model       R²          MAE         RMSE
Gradient Boosting 0.727484 5.468321e+06 8.934704e+06
    Random Forest 0.684188 5.720218e+06 9.618305e+06
      Extra Trees 0.674222 5.817081e+06 9.768890e+06


In [9]:
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

advanced_models = {
    "XGBoost": XGBRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        random_state=42,
        objective="reg:squarederror"
    ),
    "LightGBM": LGBMRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        random_state=42,
        verbosity=-1
    )
}

advanced_results = []

for name, model in advanced_models.items():
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_val)

    advanced_results.append({
        "Model": name,
        "R²": r2_score(y_val, predictions),
        "MAE": mean_absolute_error(y_val, predictions),
        "RMSE": np.sqrt(mean_squared_error(y_val, predictions))
    })

advanced_results_df = pd.DataFrame(advanced_results).sort_values(
    "R²", ascending=False
)

print(advanced_results_df.to_string(index=False))

   Model       R²          MAE         RMSE
 XGBoost 0.708133 5.961122e+06 9.246495e+06
LightGBM 0.688527 6.083489e+06 9.552003e+06


In [12]:
from sklearn.model_selection import GridSearchCV

# tune hyperparameters for XGBoost using GridSearchCV
xgb_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", XGBRegressor(
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    ))
])

param_grid = {
    "model__n_estimators": [200, 300, 500],
    "model__max_depth": [3, 4, 5],
    "model__learning_rate": [0.03, 0.05, 0.1],
    "model__subsample": [0.8, 1.0],
    "model__colsample_bytree": [0.8, 1.0]
}

grid_search = GridSearchCV(
    xgb_pipeline,
    param_grid,
    cv=3,
    scoring="r2",
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)
print("Best cross-validation R²:", grid_search.best_score_)

best_xgb = grid_search.best_estimator_
val_predictions = best_xgb.predict(X_val)

print("Validation R²:", r2_score(y_val, val_predictions))
print("Validation MAE:", mean_absolute_error(y_val, val_predictions))
print("Validation RMSE:", np.sqrt(mean_squared_error(y_val, val_predictions)))

KeyboardInterrupt: 

In [37]:
# Get feature names after preprocessing
feature_names = best_xgb.named_steps["preprocessor"].get_feature_names_out()

# Get XGBoost feature importances
importances = best_xgb.named_steps["model"].feature_importances_

importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
}).sort_values("Importance", ascending=False)

print(importance_df.head(15).to_string(index=False))

                   Feature  Importance
                  num__G+A    0.231530
num__Shot Creating Actions    0.170916
           num__Value Team    0.129589
        num__Shots Blocked    0.052786
             num__Mistakes    0.030549
                  num__Age    0.029581
                  num__Min    0.027786
  num__Dribbles Successful    0.026309
          cat__Position_MF    0.020428
          cat__Position_FW    0.018277
                   num__xA    0.018178
       cat__Position_MF,DF    0.016956
       num__Air Duels Won%    0.016893
        num__Through Balls    0.016892
                   num__xG    0.014855


In [38]:
test_predictions = best_xgb.predict(X_test)

print("Test R²:", r2_score(y_test, test_predictions))
print("Test MAE:", mean_absolute_error(y_test, test_predictions))
print("Test RMSE:", np.sqrt(mean_squared_error(y_test, test_predictions)))

Test R²: 0.6593446969301039
Test MAE: 6357190.714431533
Test RMSE: 11203481.865744315


In [39]:
# Combine validation results from all five models
all_results = results + advanced_results

comparison_df = pd.DataFrame(all_results)

comparison_df = comparison_df.sort_values(
    by="R²", ascending=False
).reset_index(drop=True)

comparison_df[["MAE", "RMSE"]] = comparison_df[
    ["MAE", "RMSE"]
].round(2)

print(comparison_df.to_string(index=False))

            Model       R²        MAE       RMSE
          XGBoost 0.708133 5961121.65 9246494.77
    Random Forest 0.701658 5986553.39 9348495.32
Gradient Boosting 0.695766 5925018.04 9440353.21
         LightGBM 0.688527 6083489.37 9552002.80
      Extra Trees 0.679510 6268309.67 9689275.06


In [40]:
print("Value Team" in final_df.columns)
print(final_df.columns.tolist())

True
['Unnamed: 0', 'Player', 'Position', 'Team', 'Comp', 'Age', 'MP', 'Min', 'Goals', 'Ast', 'G+A', 'xG', 'xA', 'SoT%', 'npxG/Sh', 'G-xG', 'Pass Cmp%', 'Through Balls', 'Switch Passes', 'Crosses', 'Shot Creating Actions', 'Def Action to Shot', 'Tackles', 'Shots Blocked', 'Interceptions', 'Clearances', 'Errors', 'Dribbles Successful', 'Dribble Distance', 'Mistakes', 'Ball Recoveries', 'Air Duels Won%', 'Values', 'Value Team']
